# Weighted Directed Graph Creation from PostGIS

This notebook adds edge weights to the navigation graph based on distance, depth, weather, and traffic. It enables realistic maritime routing by accounting for vessel constraints and environmental factors.

#### Workflow Overview

1. **Convert to Directed** - Transform undirected graph to support bidirectional routing
2. **Enrich with Depth** - Add S-57 feature data (depth, orientation, clearance) to edges
3. **Apply Static Weights** - Calculate distance-based penalties/bonuses from maritime features
4. **Apply Directional Weights** - Add traffic flow alignment penalties/rewards
5. **Apply Dynamic Weights** - Combine all tiers with vessel-specific constraints
6. **Calculate Route** - Find optimal path using final `adjusted_weight`

#### Data Flow

```
PostGIS → Directed Conversion → Edge Enrichment → Static Weights → Directional Weights → Dynamic Weights → Optimized Route
```

#### Expected Outputs

- **Directed Graph**: Graph with bidirectional edges for traffic-aware routing
- **Enriched Edges**: Features attributes (depth, orientation, clearance) added
- **Weighted Graph**: Three-tier weights combined into `adjusted_weight`
- **Optimized Route**: Path considering depth, weather, traffic, and vessel constraints
- **Benchmarks**: Performance metrics for each weighting step

#### Required Data

This notebook requires:
1. **Base Graph**: Existing navigation graph from fine graph workflow (in PostGIS)
2. **ENC Data**: S-57 charts converted to PostGIS format (with SOUNDG layer)
3. **Database Schema**: Schema containing S-57 layers (e.g., `enc_west`)
4. **Connection**: PostgreSQL credentials configured in `.env` file

**Setup Instructions:** See `docs/SETUP.md`
**Troubleshooting:** See `docs/TROUBLESHOOTING.md`

## 1. Configuration

Adjust parameters below to control the weighting pipeline. All user-configurable values are centralized here.

**Quick Start:**
- Toggle `workflow_steps` to skip completed operations
- Adjust `vessel_params` for your vessel specifications  
- Modify `env_conditions` for current weather/visibility

**Detailed parameter documentation:** See APPENDIX A.2

**Production configuration:** See `maritime_workflow_config.yml` for full parameter schemas

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION
# =============================================================================
# --- Graph Configuration ---
graph_name_undirected = "fine_graph_20" # Source (undirected) graph
graph_name_directed = "fine_graph_directed_20" # Target for directed, weighted graph

# Set True if you want to export the final graph to GeoPackage
run_export = True

# --- Data Source Configuration ---
enc_schema = "enc_west"  # Schema containing S-57 ENC data (configurable)

# --- Workflow Control ---
# Set these to True or False to enable/disable steps.
# For a full run, set all to True. For a partial run, disable completed steps.
workflow_steps = {
    "run_conversion_to_directed": True, # Creates the new directed graph tables
    "run_enrichment": True,             # Adds S-57 feature data (ft_*) to edges. REQUIRED for all weighting.
    "run_static_weights": True,         # Applies weights from static layers (land, fairways, etc.)
    "run_directional_weights": True,    # Applies weights based on traffic flow (TSS, fairways)
    "run_dynamic_weights": True,        # Applies final vessel-specific weights (draft, height)
    "run_pathfinding": True             # Loads the final graph and calculates a route
}

# --- Vessel & Environment Parameters (for Dynamic Weights & Pathfinding) ---
vessel_params = {
 'draft': 7.5,           # meters
 'height': 30.0,         # meters (for overhead clearance)
 'safety_margin': 2.0,   # meters (for under-keel clearance)
 'vessel_type': 'cargo'
}

env_conditions = {
 'weather_factor': 1.2,      # 1.0=good, >1.0=poor
 'visibility_factor': 1.1,   # 1.0=good, >1.0=poor
 'time_of_day': 'day'        # 'day' or 'night'
}

# --- Pathfinding Ports ---
departure_port_name = "SF Pilot"
arrival_port_name = "San Francisco Arrival"

departure_coords = {"lon": -122.780, "lat": 37.006}
arrival_coords = {"lon": -122.400, "lat": 37.805}

### 1.2 Imports & Environment Setup

This section imports all necessary libraries and initializes the core classes for the weighting pipeline:

- **ENCDataFactory**: Provides unified interface for accessing S-57 ENC data from PostGIS
  - Handles database connections and layer queries
  - Used by all other classes for data access

- **H3Graph & FineGraph**: Manage graph operations (conversion, loading, saving, export)
  - Convert undirected graphs to directed
  - Load/save graphs from/to PostGIS tables
  - This notebook uses generic graph operations that work with **both fine grids and H3 graphs**
  - The specific graph type doesn't affect weighting - only the input table name matters

- **Weights**: Implements the three-tier weighting system (static, directional, dynamic)
  - Static weights: Distance-based penalties/bonuses from features
  - Directional weights: Traffic flow alignment penalties/rewards
  - Dynamic weights: Vessel-specific constraints (draft, height) combined with environmental conditions
  - Combines all tiers into final `adjusted_weight`

- **Route**: Calculates optimal routes on weighted graphs using A* algorithm

- **PortData**: Manages port definitions (World Port Index + custom ports)

The output directory is created here for saving routes and benchmarks. Database credentials are loaded from the `.env` file (secure, not hardcoded). All processing is performed server-side in PostGIS for maximum performance and scalability.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import time
import geopandas as gpd
import pandas as pd
import plotly.express as px
from shapely.geometry import Point

# --- Fix PROJ_LIB Path (Common Conda/Jupyter Issue) ---
# Ensure GDAL/PROJ can find the coordinate database
conda_prefix = sys.prefix
possible_proj_lib = os.path.join(conda_prefix, 'share', 'proj')
if os.path.exists(possible_proj_lib):
    os.environ['PROJ_LIB'] = possible_proj_lib

# --- Setup Python Environment ---
# Add the project root to the Python path to enable module imports
project_root = Path.cwd().parent.parent


# --- Import Maritime Module Components ---
from nautical_graph_toolkit.core.graph import H3Graph, Weights, FineTuning
from nautical_graph_toolkit.core.s57_data import ENCDataFactory
from nautical_graph_toolkit.core.pathfinding_lite import Route
from nautical_graph_toolkit.utils.port_utils import PortData
from nautical_graph_toolkit.utils.notebook_utils import BenchmarkLogger, load_estimates

# Load environment variables from .env file at the project root
# This loads database credentials and API tokens
load_dotenv(project_root / ".env")

# --- Define Output Directory ---
output_dir = Path.cwd() / 'output'
output_dir.mkdir(exist_ok=True)

# --- Database Connection Parameters ---
# PostGIS connection parameters loaded from environment variables
db_params = {
    'dbname': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT')
}

# --- Initialize Core Classes ---
# ENCDataFactory: Interface for S-57 ENC data access (database connections, layer queries)
# graph: Generic graph management (works with fine grids, H3, and other graph types)
# weights_manager: Three-tier weighting system implementation
# port_manager: Port location management (World Port Index + custom ports)
# See APPENDIX A.5 for detailed class architecture and responsibilities.

factory = ENCDataFactory(source=db_params, schema=enc_schema)
graph = H3Graph(data_factory=factory, route_schema_name="routes", graph_schema_name="graph")
weights_manager = Weights(data_factory=factory)
port_manager = PortData()

# --- Initialize Performance Tracking ---
# Use BenchmarkLogger for unified performance tracking across all notebooks
logger = BenchmarkLogger()
logger.configure_weighted_graph(
    vessel_draft_m=vessel_params['draft'],
    vessel_height_m=vessel_params['height'],
    vessel_type=vessel_params['vessel_type'],
    weather_factor=env_conditions['weather_factor'],
    enc_count=None
)

# Set workflow metadata
logger.set_result('workflow', 'graph_weighted_directed_postgis_v2')
logger.set_result('data_source', 'PostGIS')
logger.set_result('graph_name', graph_name_directed)
logger.set_result('db_schema', 'graph')

print("Setup complete. Core classes initialized.")
print(f"📊 ENC schema: {enc_schema}")

### 1.3 Workflow Context

  **Pipeline Position**: Step 3 of 3 (Final routing optimization)
  1. **Data Import** (`import_s57.ipynb`) - Convert S-57 ENCs to PostGIS/GeoPackage/SpatiaLite
  2. **Graph Construction** (`graph_fine_*_v2.ipynb` or `graph_*_v2.ipynb`) - Build navigation graph
  3. **Weighting & Routing** (This notebook) - Add edge weights and compute optimized routes

  **Prerequisites**:
  - Completed fine graph creation (`graph_fine_PostGIS_v2.ipynb`) with directed graph tables
  - S-57 ENC data imported to PostGIS with soundings (SOUNDG layer) for depth enrichment
  - Database credentials configured in `.env` file
  - Vessel parameters defined (draft, height, type)

  **Outputs**:
  - Directed, weighted graph with three-tier weights (`adjusted_weight` column on edges)
  - Enriched edges with S-57 feature attributes (ft_depth, ft_orient, ft_verclr, etc.)
  - Optimized maritime route accounting for vessel constraints and environmental factors
  - Performance benchmarks appended to `benchmark_graph_weighted_directed.csv`
  - Route exported to GeoJSON for visualization in QGIS or other GIS tools

  **Next Steps**:
  - Visualize route in QGIS using exported GeoJSON file
  - Experiment with different vessel parameters to understand routing sensitivity
  - Run `maritime_graph_postgis_workflow.py` for automated pipeline on other regions
  - Use routes in downstream applications (ETA calculation, fuel estimation, AIS validation)
  - Consider switching backends if performance requirements change (PostGIS for production, GeoPackage for portable/offline use)

## 2. Determine Relevant ENCs

Identify which Electronic Navigational Charts (ENCs) overlap with the graph area. This list is used throughout the workflow for feature enrichment and weight calculations.

In [ ]:
# --- Determine ENC List for the entire workflow ---
# This is done once at the start for efficiency and consistency across all steps.
# The ENC list defines which charts are relevant for the graph area and will be
# used for enrichment and weight calculations.
print("\nDetermining relevant ENCs from the source graph boundary...")
try:
    # Use the original undirected graph to define the geographic scope
    # by creating a convex hull around all graph nodes
    nodes_df_undirected = gpd.read_postgis(
        f'SELECT geometry FROM graph."{graph_name_undirected}_nodes"',
        factory.manager.engine,
        geom_col='geometry'
    )
    graph_boundary = nodes_df_undirected.geometry.union_all().convex_hull
    enc_list = factory.get_encs_by_boundary(graph_boundary)
    print(f"Found {len(enc_list)} ENCs for this workflow.")
    if not enc_list:
        print("Warning: No ENCs found for the graph boundary. Subsequent steps may fail.")
except Exception as e:
    print(f"Could not determine ENC list from source graph '{graph_name_undirected}'. Error: {e}")
    enc_list = []  # Ensure enc_list exists to avoid errors

logger.set_result("enc_list", len(enc_list) if 'enc_list' in locals() else 0)

## 3. Convert to Directed Graph

**Why this step is needed:** The undirected fine/H3 graph treats edges bidirectionally. However, maritime traffic often has directional rules:
- One-way traffic lanes (Traffic Separation Schemes)
- Current/wind patterns that favor certain directions
- Fairway orientation preferences

Converting to a directed graph allows us to:
1. Assign different weights to forward and reverse directions
2. Model one-way traffic lanes and channels
3. Apply directional bonuses/penalties based on traffic flow
4. Prepare for sophisticated routing that considers real-world constraints

**How it works:** Each undirected edge (A-B) becomes two directed edges (A→B and B→A). Feature data is propagated to both directions during enrichment. The operation is performed entirely on the database side for maximum performance.

In [ ]:
if workflow_steps["run_conversion_to_directed"]:
    logger.start_timer('conversion_to_directed')
    print(f"Converting '{graph_name_undirected}' to directed graph '{graph_name_directed}'...")

    graph.convert_to_directed_postgis(
        source_table_prefix=graph_name_undirected,
        target_table_prefix=graph_name_directed,
        edges_schema="graph",
        drop_existing=True  # Ensures a clean start for the pipeline
    )

    elapsed = logger.end_step('conversion_to_directed')
    print(f"✓ Conversion complete in {elapsed:.2f}s")
else:
    print("Skipping conversion to directed graph.")

## 4. Enrich Edges with S-57 Features

This is a **critical prerequisite** for all weighting steps. Enrichment performs spatial joins to extract navigational data from S-57 ENC layers and attach it to graph edges.

### What Gets Enriched

The enrichment process adds `ft_*` columns to edges for each relevant S-57 feature:
- **Depth data** (`ft_depth`, `ft_valsou`): Water depth from soundings (for draft clearance checks)
- **Orientation** (`ft_orient`, `ft_trafic`): Traffic flow direction from TSS/fairways (for directional weights)
- **Clearance** (`ft_verclr`): Vertical clearance from bridges/cables (for height checks)
- **Feature presence** (`ft_lndare`, `ft_fairwy`, `ft_tsslpt`, etc.): Which feature types affect each edge
- **Safety characteristics**: Sounding values, construction types, precautionary area flags

### Why This is Required

All downstream weighting steps depend on this enriched data:
- **Static weights** need feature layers (`ft_lndare`, `ft_fairwy`, etc.) to calculate distance-based penalties/bonuses
- **Directional weights** need `ft_orient` and `ft_trafic` to align routes with traffic flow
- **Dynamic weights** need `ft_depth` and `ft_verclr` to enforce vessel-specific constraints

### Performance Note

This step can take 10-30 minutes for large graphs but only needs to be run once. All feature data is stored permanently in the edge table for subsequent weight calculations and is reusable for any routing scenario.

### Data Propagation

When `is_directed=True`, features are propagated to reverse edges to ensure bidirectional data consistency. This means:
- Forward edge (A→B) gets features from feature geometries
- Reverse edge (B→A) gets the same features
- Directional properties can then be applied differently to each direction during directional weighting

### ✓ PostGIS Performance Note

PostGIS is **optimized for spatial operations** and performs comparably to the Python in-memory approach:

**Real Performance Measurements (361,058 edges, 15 ENC layers):**
- Python in-memory approach (`apply_static_weights()`): ~2.3-2.4 minutes
- PostGIS server-side (`apply_static_weights_postgis()`): ~2.3 minutes ✓ **EXCELLENT**

**Why PostGIS performs well:**
- PostGIS has highly optimized spatial query engines (much better than SpatiaLite)
- Server-side processing avoids data transfer overhead
- Spatial indexes are efficiently utilized
- No significant per-layer overhead like with SpatiaLite

**Note:** If you were using GeoPackage instead of PostGIS, the same function would be 5-10x slower due to SpatiaLite limitations. PostGIS is the recommended production choice for large-scale maritime routing.

In [ ]:
if workflow_steps["run_enrichment"]:
    logger.start_timer('edge_enrichment')
    try:
        # Get the feature layers to extract from the S57 classifier
        # This defines which S-57 object classes will be joined to edges
        # (e.g., seaare, lndare, fairwy, tsslpt, soundg, etc.)
        feature_layers_to_enrich = weights_manager.get_feature_layers_from_classifier()

        # Run the server-side enrichment process
        # This performs spatial joins entirely in PostGIS for maximum performance:
        # - For each feature layer, finds edges that intersect or are near features
        # - Extracts relevant attributes (depth, orientation, clearance, etc.)
        # - Adds ft_* columns to the edges table with the extracted data
        # - Propagates data to reverse edges when is_directed=True
        print(f"Enriching {len(enc_list)} ENCs across {len(feature_layers_to_enrich)} feature layers...")
        summary = weights_manager.enrich_edges_with_features_postgis(
            enc_names=enc_list,  # ENCs determined from graph boundary in setup
            schema_name="graph",
            graph_name=graph_name_directed,
            enc_schema=enc_schema,
            feature_layers=feature_layers_to_enrich,
            is_directed=True,  # IMPORTANT: Ensures features are propagated to reverse edges
            include_sources=False,  # Don't store ENC source names (saves space)
            soundg_buffer_meters=30  # Buffer for sounding point queries (depth data)
        )
        elapsed = logger.end_step('edge_enrichment')
        print(f"\n✓ Enrichment complete in {elapsed:.2f}s")
        print("\nFeatures extracted:")
        for feature, count in summary.items():
            print(f"  • {feature}: {count:,} edges")

    except Exception as e:
        print(f"✗ An error occurred during enrichment: {e}")
        print("Please ensure the directed graph tables exist.")
else:
    print("⊘ Skipping edge enrichment")

## 5. Apply Static Weights

This step applies **distance-based penalties and bonuses** from static S-57 features. The three-tier system categorizes features as:

### Three-Tier Weight Categories

1. **Blocking Weights** (`wt_static_blocking`): Absolute avoidance zones
   - Land areas (`lndare`): factor = 100
   - Underwater rocks (`uwtroc`): factor = 100
   - Shoreline constructions (`slcons`): factor = 90
   - Edges within these features become effectively impassable

2. **Penalty Weights** (`wt_static_penalty`): Areas to avoid when possible
   - Currently not used in this configuration
   - Can be configured for areas like anchorages, restricted zones

3. **Bonus Weights** (`wt_static_bonus`): Preferred routing areas
   - Fairways (`fairwy`): factor = 0.5 (50% cost reduction)
   - Traffic Separation Schemes (`tsslpt`): factor = 0.7
   - Dredged areas (`drgare`): factor = 0.9
   - Precautionary areas (`prcare`): factor = 0.9
   - Recommended tracks (`rectrc`, `dwrtcl`): factor = 0.5

### Distance-Based Degradation

Weights are applied using a **distance degradation model** defined in `graph_config.yml`:
- Features have an influence `buffer` (e.g., 500m for underwater rocks)
- Weight intensity decreases with distance from feature
- Edges far from features get neutral weight (1.0)
- Edges near/within features get the configured factor

This allows smooth transitions between safe and dangerous areas rather than hard boundaries.

In [ ]:
if workflow_steps["run_static_weights"]:
    logger.start_timer('static_weights')
    print("Applying static weights...")

    # Load configuration to get static layer definitions
    # graph_config.yml defines which layers are blocking/penalty/bonus
    # and the distance degradation parameters for each
    config = weights_manager._load_config()

    # Apply static weights using server-side PostGIS operations
    # This creates/updates three columns: wt_static_blocking, wt_static_penalty, wt_static_bonus
    # Each edge gets weights based on its spatial relationship to features

    weights_manager.apply_static_weights_postgis(
        graph_name=graph_name_directed,
        enc_names=enc_list,  # Use the ENCs determined in setup
        schema_name="graph",
        enc_schema=enc_schema,
        static_layers=config["weight_settings"]["static_layers"],
        usage_bands=[3, 4, 5]  # Focus on higher-detail bands for static features
                               # Band 3: Approach (1:90K), Band 4: Harbour (1:22K-45K)
                               # Band 5: Berthing (1:4K-12K)
    )

    elapsed = logger.end_step('static_weights')
    print(f"✓ Static weights applied in {elapsed:.2f}s")
else:
    print("Skipping static weight application.")

## 6. Apply Directional Weights

This step calculates **traffic flow alignment penalties and rewards** based on how well an edge aligns with the intended direction of maritime traffic features.

#### How Directional Weights Work

1. **Uses enriched orientation data** (`ft_orient` and `ft_trafic` from enrichment step):
   - `ft_orient`: The intended traffic direction (0-360 degrees) from features like TSS lanes and fairways
   - `ft_trafic`: Traffic direction code (1=one-way, 2=two-way)

2. **Calculates angular difference** between:
   - Edge direction (from start node to end node)
   - Feature orientation (intended traffic flow)
   - Converted to 0-360 degree format for comparison

3. **Applies penalties/rewards based on alignment**:
   - **Aligned** (0-22.5°): Weight = 0.7 (30% cost reduction for following traffic)
   - **Slightly off** (22.5-45°): Weight = 1.2 (20% penalty)
   - **Moderately off** (45-90°): Weight = 5.0 (substantial penalty)
   - **Opposite direction** (135-180°): Weight = 50.0 (strongly discouraged)

#### Two-Way Traffic Handling

**Two-way traffic handling**:
   - For two-way lanes (`ft_trafic=2`), checks if reverse edge is well-aligned
   - If reverse edge alignment > 95%, allows travel in both directions
   - Prevents penalizing legitimate two-way routes

#### Affected Layers

Directional weights are applied to:
- **Traffic Separation Schemes** (`tsslpt`): One-way shipping lanes
- **Fairways** (`fairwy`): Main navigation channels
- **Recommended tracks** (`rectrc`, `dwrtcl`): Preferred routes

This ensures routes follow established maritime traffic patterns and avoid wrong-way travel in one-way lanes.

In [ ]:
if workflow_steps["run_directional_weights"]:
    logger.start_timer('directional_weights')
    print("Applying directional weights...")

    # Load configuration for angle bands and layer settings
    # graph_config.yml defines:
    # - Angle bands with thresholds and weight factors
    # - Layers to apply directional weights to
    # - Two-way traffic detection parameters
    config = weights_manager._load_config()
    directional_config = config["weight_settings"]["directional_weights"]
    two_way_config = directional_config.get("two_way_traffic", {})

    # Calculate directional weights using server-side PostGIS operations
    # This creates/updates the wt_dir column based on:
    # 1. Edge geometry direction (azimuth from start to end node)
    # 2. Feature orientation from ft_orient (extracted during enrichment)
    # 3. Traffic direction from ft_trafic (1=one-way, 2=two-way)
    summary = weights_manager.calculate_directional_weights_postgis(
        schema_name="graph",
        graph_name=graph_name_directed,
        apply_to_layers=directional_config.get("apply_to_layers"),  # tsslpt, fairwy, rectrc, dwrtcl
        angle_bands=directional_config.get("angle_bands"),  # Angle thresholds and weights
        two_way_enabled=two_way_config.get("enabled", True),  # Enable two-way detection
        reverse_check_threshold=two_way_config.get("reverse_check_threshold", 95)  # 95% alignment
    )
    elapsed = logger.end_step('directional_weights')
    print(f"\n✓ Directional weights applied in {elapsed:.2f}s")
    print(f"  • Edges updated: {summary['edges_updated']:,}")
    print(f"  • Edges with orientation: {summary['edges_with_orient']:,}")
    print(f"  • Rewarded: {summary['edges_rewarded']:,}")
    print(f"  • Small penalty: {summary['edges_small_penalty']:,}")
    print(f"  • Moderate penalty: {summary['edges_moderate_penalty']:,}")
    print(f"  • High penalty: {summary['edges_high_penalty']:,}")
    print(f"  • Opposite: {summary['edges_opposite']:,}")
else:
    print("Skipping directional weight application.")

## 7. Apply Dynamic (Vessel-Specific) Weights

This is the **final weighting step** that combines all previous weights with vessel-specific constraints to produce the `adjusted_weight` used for pathfinding.

#### Three-Tier Integration

Dynamic weights integrate all three tiers from previous steps:

##### Tier 1: Blocking Factor
Combines static blocking weights with vessel physical constraints:
- **Under-Keel Clearance (UKC)**: Checks if water depth (`ft_depth`) is sufficient for vessel draft + safety margin
- **Vertical Clearance**: Checks if bridge clearance (`ft_verclr`) exceeds vessel height
- **Result**: Shallow water or low bridges get extremely high weights (effectively blocked)

##### Tier 2: Penalty Factor
Combines static penalties with environmental conditions:
- Weather degradation (e.g., 1.2x penalty for poor weather)
- Visibility reduction (e.g., 1.1x penalty for poor visibility)
- Time-of-day adjustments (e.g., night navigation penalties)
- **Result**: Moderate weight increases for less favorable conditions

##### Tier 3: Bonus Factor
Uses static bonus weights with vessel type preferences:
- Fairways and TSS lanes (from static weights)
- Deep water channels (preferred by large vessels)
- **Result**: Weight reductions for preferred routes


#### Final Weight Calculation

The `adjusted_weight` for each edge is calculated as:
```
adjusted_weight = base_distance × blocking_factor × penalty_factor × bonus_factor × directional_weight
```

Where:
- `base_distance`: Original edge length in nautical miles (stored in `weight` column)
- `blocking_factor`: From `wt_static_blocking` + UKC/clearance checks
- `penalty_factor`: From `wt_static_penalty` + environmental conditions
- `bonus_factor`: From `wt_static_bonus` + vessel preferences
- `directional_weight`: From `wt_dir` (traffic flow alignment)

#### Result

Edges that are **safe, aligned with traffic, in preferred areas, and suitable for the vessel** get low weights (preferred routes). Edges that are **dangerous, misaligned, or unsuitable** get high weights (avoided routes).

In [ ]:
if workflow_steps["run_dynamic_weights"]:
    logger.start_timer('dynamic_weights')
    print("Calculating final dynamic weights...")

    # This step performs the final weight integration entirely in PostGIS:
    # 1. Reads all previously calculated weight columns (wt_static_*, wt_dir)
    # 2. Reads enriched feature attributes (ft_depth, ft_verclr, etc.)
    # 3. Applies vessel constraint checks (draft vs depth, height vs clearance)
    # 4. Applies environmental condition factors (weather, visibility, time)
    # 5. Combines all factors into the final adjusted_weight column
    # 6. Preserves original 'weight' column (distance) for reference
    summary = weights_manager.calculate_dynamic_weights_postgis(
        graph_name=graph_name_directed,
        schema_name="graph",
        vessel_parameters=vessel_params,  # From configuration cell: draft, height, type
        environmental_conditions=env_conditions,  # From configuration: weather, visibility, time
    )
    elapsed = logger.end_step('dynamic_weights')

    print(f"✓ Dynamic weights calculated in {elapsed:.2f}s")
    print(f"  • Edges updated: {summary['edges_updated']:,}")
    print("IMPORTANT: The 'adjusted_weight' column now contains the final routing weights.")
    print("           Use weight_key='adjusted_weight' in pathfinding operations.")
else:
    print("Skipping dynamic weight calculation.")

## 8. Pathfinding and Analysis

With the graph fully weighted, calculate an optimal route between departure and arrival points using the `adjusted_weight` that incorporates all weight tiers. The route is then saved to a file for visualization in a GIS application.

### 8.1. Load Weighted Graph

This step loads the final, fully weighted graph from PostGIS into an in-memory `networkx` object. This can be time-consuming for large graphs.

In [ ]:
if workflow_steps["run_pathfinding"]:
    logger.start_timer('graph_loading')
    print(f"--- Loading final weighted graph '{graph_name_directed}' from PostGIS... ---")
    try:
        G = graph.load_graph_from_postgis(graph_name_directed)
        elapsed = logger.end_step('graph_loading')
        print(f"✓ Graph loaded in {elapsed:.2f}s")
        print(f"  • Nodes: {G.number_of_nodes():,}")
        print(f"  • Edges: {G.number_of_edges():,}")
        
        # Store graph statistics
        logger.set_result('node_count', G.number_of_nodes())
        logger.set_result('edge_count', G.number_of_edges())
    except Exception as e:
        print(f"Failed to load graph: {e}")
        G = None # Ensure G is None on failure
else:
    print("⊘ Skipping pathfinding step, graph will not be loaded.")
    G = None

### 8.2. Calculate and Save Route

Using the loaded graph, this step calculates the optimal route between the specified departure and arrival points using the final `adjusted_weight`.

In [ ]:
if workflow_steps["run_pathfinding"] and G is not None:
    logger.start_timer('route_calculation')
    print("\n--- Starting Route Calculation ---")

    # Create or update the custom port locations
    port_manager.create_custom_port(port_name=departure_port_name,
                                    lon=departure_coords['lon'],
                                    lat=departure_coords['lat'],
                                    if_exists='update'
                                    )
    port_manager.create_custom_port(port_name=arrival_port_name,
                                    lon=arrival_coords['lon'],
                                    lat=arrival_coords['lat'],
                                    if_exists='update'
                                    )

    # Get port geometries
    departure_port = port_manager.get_port_by_name(departure_port_name)
    arrival_port = port_manager.get_port_by_name(arrival_port_name)

    if departure_port.empty or arrival_port.empty:
        print("Error: Could not find departure or arrival port.")
    else:
        # Initialize the routing engine with the loaded graph
        route_finder = Route(graph=G, data_manager=factory.manager)

        # Calculate the detailed route
        print(f"Calculating route from '{departure_port_name}' → '{arrival_port_name}'...")
        route_detail = route_finder.detailed_route(
            departure_point=departure_port.geometry,
            arrival_point=arrival_port.geometry,
            weight_key='adjusted_weight' # CRITICAL: Use the final calculated weight
        )

        # Save the route to a file for visualization
        output_path = output_dir / f"detailed_route_{vessel_params['draft']}m_draft.geojson"
        route_finder.save_detailed_route_to_file(route_detail,
                                                 output_path=str(output_path))

        elapsed = logger.end_step('route_calculation')
        print(f"\n✓ Route calculated in {elapsed:.2f}s")
        print(f"✓ Route exported to: {output_path}")

elif workflow_steps["run_pathfinding"] and G is None:
    print("Skipping route calculation because the graph failed to load.")
else:
    print("⊘ Skipping route calculation")

## 9. (Optional) Export Weighted Graph

If you need to use the final weighted graph in another application (like QGIS), you can export it from PostGIS to a GeoPackage file. This is much more efficient than loading it into memory first.

In [ ]:
if run_export:
    output_gpkg_path = output_dir / f"{graph_name_directed}.gpkg"
    print(f"Exporting '{graph_name_directed}' from PostGIS to '{output_gpkg_path}'...")

    # Use the efficient ogr2ogr-based export method
    try:
        graph.export_postgis_to_gpkg(
            schema_name="graph",
            graph_name=graph_name_directed,
            output_path=str(output_gpkg_path)
        )
        print("Export complete.")
    except FileExistsError:
        print(f"Export failed: File already exists at {output_gpkg_path}. Please delete it and try again.")
    except Exception as e:
        print(f"An error occurred during export: {e}")
else:
    print("Skipping graph export.")

## 10. Workflow Summary and Next Steps

Congratulations! You've completed the weighting and pathfinding pipeline. Here's what was accomplished:

### What You've Created

1. **Directed Graph** (`h3_graph_directed_pg_6_11`): Undirected graph converted to support directional routing
2. **Enriched Edges**: All edges now have S-57 feature attributes (depth, orientation, clearance, etc.)
3. **Three-Tier Weights**: Edges have static, directional, and dynamic weights combined into `adjusted_weight`
4. **Optimal Route**: A route computed using the final weighted graph that balances:
   - Safe passage (avoiding land, shallow areas, overhead hazards)
   - Traffic compliance (following fairways, TSS lanes, recommended tracks)
   - Vessel constraints (draft, height, type-specific preferences)
   - Environmental factors (weather, visibility, time of day)

### Understanding the Weights

The final `adjusted_weight` on each edge combines:
```
adjusted_weight = base_distance × blocking_factor × penalty_factor × bonus_factor × directional_weight
```

Where:
- **base_distance**: Original edge length (nautical miles)
- **blocking_factor**: Absolute obstacles (land, shallow water) - high = impassable
- **penalty_factor**: Areas to avoid (environmental conditions, hazards)
- **bonus_factor**: Preferred areas (fairways, TSS lanes, dredged channels) - <1.0 = encouraged
- **directional_weight**: Traffic flow alignment (follow one-way lanes, align with fairways)

### Next Steps

**For Further Analysis:**
- Examine route segments in QGIS to understand routing decisions
- Compare routes with different vessel parameters (draft, height)
- Analyze weight distributions to identify bottleneck areas
- Validate against real-world maritime practices

**For Production Use:**
- Use `maritime_graph_postgis_workflow.py` for automated full pipeline (Steps 1-4)
- Configure `maritime_workflow_config.yml` with your specific parameters
- Integrate routes into navigation systems, ETA calculators, or fuel estimation tools
- Update weighting factors based on operational experience and feedback

**For Performance Optimization:**
- Review benchmark metrics (`benchmark_graph_weighted_directed.csv`) to identify slow steps
- Consider using optimized save methods for future large graphs
- Experiment with different vessel parameters to understand weight sensitivities
- Profile the weighting steps that take longest for your specific AOI

### Benchmark Results

See the performance summary above for timing data on each step. Key metrics:
- **Edge Enrichment**: Usually the longest step (10-30 min) but runs once
- **Directional Weights**: Depends on feature coverage (5-15 min)
- **Dynamic Weights**: Combines all factors (~5-10 min)
- **Graph Loading**: Only needed for in-memory pathfinding (~10 min for large graphs)
- **Route Calculation**: Fast once graph is loaded (<1 min)

## Performance Summary

Save detailed performance metrics to CSV for long-term tracking and comparison across different configurations.

In [ ]:
# --- Export Benchmark to CSV ---
csv_path = logger.export_benchmark()
print(f"\n💾 Benchmark saved to: {csv_path}")

# --- Display Benchmark Summary ---
print("\n")
print(logger.get_current_benchmark_summary(csv_path))

# --- Visualize Pipeline Performance ---
print("\n")
fig = logger.visualize_performance(
    title='Weighted Directed Graph Pipeline Performance (PostGIS)',
    sort_by='time_descending',
    show=True
)

## APPENDIX: Detailed Documentation

### A.1 Understanding the Three-Tier Weight System

**Weight Formula:**
```
adjusted_weight = base_distance × blocking_factor × penalty_factor × bonus_factor × directional_weight
```

**Tier 1: Blocking Factor** (static_blocking + vessel constraints)
- Land areas, underwater rocks, shallow water (depth < draft + safety margin)
- Bridges with insufficient clearance (height > clearance)
- Result: Extremely high weights (100x) = effectively impassable

**Tier 2: Penalty Factor** (static_penalty + environmental conditions)
- Weather degradation (1.2x for poor weather)
- Visibility reduction (1.1x for fog/night)
- Time-of-day adjustments
- Result: Moderate weight increases (1.2-5.0x)

**Tier 3: Bonus Factor** (static_bonus + vessel preferences)
- Fairways: 0.5x (50% cost reduction)
- TSS lanes: 0.7x (30% reduction)
- Dredged areas: 0.9x (10% reduction)
- Result: Weight reductions for preferred routes

**Directional Component:**
- Aligned with traffic (0-30°): 0.8x (20% faster)
- Moderately misaligned (30-150°): 1.5x (50% slower)
- Opposite direction (150-210°): 3.0x (strongly discouraged)

### A.2 Parameter Tuning Guide

**Vessel Draft (`vessel_params['draft']`)**
- Definition: Vessel depth below waterline in meters
- Impact: Determines minimum navigable water depth
- Safety margin: Automatically added (2.0m → 2.64m adjusted)
- Example: 7.5m draft requires 10.14m minimum depth for safe passage

**Vessel Height (`vessel_params['height']`)**
- Definition: Height above waterline to highest point in meters
- Impact: Determines clearance under bridges/cables
- Example: 30m height cannot pass under 25m clearance bridges

**Weather Factor (`env_conditions['weather_factor']`)**
- Range: 1.0 (good) to 2.0 (severe)
- Impact: Multiplicative penalty on all edges
- Example: 1.2x increases route time by 20% in poor weather

**Usage Bands (`usage_bands` in static weights)**
- Band 3: Approach (1:90K scale) - Regional planning
- Band 4: Harbour Entrance (1:22K-45K) - Coastal navigation
- Band 5: Harbour (1:4K-12K) - Detailed harbor operations
- Recommendation: [3, 4, 5] for coastal routes, [4, 5] for harbor-only

### A.3 PostGIS-Specific Considerations

**Schema Management:**
- All graph tables stored in `graph` schema
- ENC data stored in configurable schema (`enc_schema` parameter)
- Edges table: `graph.{graph_name}_edges`
- Nodes table: `graph.{graph_name}_nodes`

**Performance Optimization:**
- Server-side spatial operations (2.0-2.4× faster than GeoPackage)
- R-tree spatial indexes on geometry columns
- Connection pooling via SQLAlchemy
- Parallel enrichment possible (not implemented in this notebook)

**Database Requirements:**
- PostgreSQL 16+ with PostGIS 3.4+
- Recommended: 8GB+ RAM for graphs >200K nodes
- SSD storage recommended for large-scale enrichment

### A.4 Production Deployment

**Workflow Integration:**
- Use `maritime_graph_postgis_workflow.py` for automated full pipeline
- Configure `maritime_workflow_config.yml` with production parameters
- Schedule periodic weight updates for dynamic conditions (weather, traffic)

**Route Storage:**
- Export routes to GeoJSON for GIS visualization
- Store routes in PostGIS `routes` schema for historical analysis
- Include metadata: vessel parameters, weather conditions, timestamps

**Monitoring:**
- Track benchmark metrics for performance regression detection
- Alert on enrichment failures (missing S-57 layers, spatial index issues)
- Monitor database size growth (enriched graphs ~5× larger than base)

**Optimization Strategies:**
- Cache enriched graphs by vessel class (small/medium/large)
- Pre-compute weights for common weather scenarios
- Use H3 graphs for multi-resolution routing (not covered in this notebook)
- Consider regional graph subdivision for very large areas (>1M nodes)

**Quality Assurance:**
- Validate routes against real-world AIS data
- Compare weighted vs unweighted route differences
- Test edge cases: shallow draft, extreme weather, night routing
- Review blocking edges to ensure no false positives (legitimate passages blocked)

### A.5 Core Class Architecture

This notebook uses four key classes from the maritime toolkit:

**ENCDataFactory**
- **Purpose**: Unified interface for accessing S-57 ENC data from PostGIS
- **Key Responsibilities**:
  - Opens database connections to PostGIS
  - Queries ENC metadata and layer information
  - Filters ENCs by geographic boundary (used in setup phase)
  - Provides layer data to weights manager for enrichment
- **Usage**: Instantiated once at setup; used by all other classes
- **Initialization**: `ENCDataFactory(source=db_params, schema=enc_schema)`

**H3Graph (or FineTuning for fine grids)**
- **Purpose**: Generic graph operations that work with any grid type (H3, fine grids, etc.)
- **Key Responsibilities**:
  - Converts undirected graphs to directed (preserves structure, creates reverse edges)
  - Loads graphs from PostGIS into NetworkX for in-memory routing
  - Exports graphs to GeoPackage format
  - Manages graph tables in PostGIS schema
- **Usage**: Single instance manages all graph operations throughout the pipeline
- **Initialization**: `H3Graph(data_factory=factory, route_schema_name="routes", graph_schema_name="graph")`

**Weights**
- **Purpose**: Implements three-tier weight system (static, directional, dynamic)
- **Key Responsibilities**:
  - Enriches edges with S-57 feature attributes (ft_* columns)
  - Calculates static weights from land, fairways, TSS, and other layers
  - Calculates directional weights based on traffic flow alignment
  - Calculates dynamic weights combining vessel constraints and environmental factors
  - Applies all weights server-side in PostGIS for performance
- **Usage**: Instantiated once; methods called sequentially for each weight tier
- **Initialization**: `Weights(data_factory=factory)`

**PortData**
- **Purpose**: Manages port definitions and locations
- **Key Responsibilities**:
  - Provides World Port Index port locations (standardized)
  - Allows creation/updating of custom ports (e.g., specific dock locations)
  - Returns port geometries for route start/end points
  - Merges standard and custom ports into unified dataset
- **Usage**: Simple queries to find ports by name or get all ports
- **Initialization**: `PortData()` (no parameters needed)

**BenchmarkLogger**
- **Purpose**: Unified performance tracking and visualization
- **Key Responsibilities**:
  - Times each major step (enrichment, weighting, pathfinding, etc.)
  - Records metadata (vessel params, graph size, data source)
  - Exports benchmark data to CSV for trend analysis
  - Visualizes performance with interactive charts
- **Usage**: Called before/after major steps to track timing
- **Initialization**: `BenchmarkLogger()` (configured per workflow)